# Desenvolvimento de módulo [Python](https://www.python.org/) para n-body gravitacional com energia;

Autor: Raphael Figueiredo Secchin

Data: 27/07/2026

---
## <font color="#1C77C3" >Informações de Sistema</font>

Nome do sistema operacional, nome do computador na rede, versão do kernel, arquitetura de hardware (32/64 bits), etc :

In [1]:
!uname -a

Linux rfsDebian 6.12.96+deb13-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.12.96-1 (2026-07-20) x86_64 GNU/Linux


Nome e versão do sistema operacional :

In [2]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Debian
Description:	Debian GNU/Linux 13 (trixie)
Release:	13
Codename:	trixie


Diversas informações da CPU, como nome do processador, frequência em MHz da CPU, número de núcleos/cores, número de threads, memória cache, arquitetura de hardware (32/64 bits), etc :

In [3]:
!lscpu

Arquitetura:                  x86_64
  Modo(s) operacional da CPU: 32-bit, 64-bit
  Tamanhos de endereço:       39 bits physical, 48 bits virtual
  Ordem dos bytes:            Little Endian
CPU(s):                       12
  Lista de CPU(s) on-line:    0-11
ID de fornecedor:             GenuineIntel
  Nome do modelo:             12th Gen Intel(R) Core(TM) i5-12450HX
    Família da CPU:           6
    Modelo:                   151
    Thread(s) per núcleo:     2
    Núcleo(s) por soquete:    8
    Soquete(s):               1
    Step:                     2
    CPU(s) scaling MHz:       66%
    CPU MHz máx.:             4400,0000
    CPU MHz mín.:             800,0000
    BogoMIPS:                 5376,00
    Opções:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr p
                              ge mca cmov pat pse36 clflush dts acpi mmx fxsr ss
                              e sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm c
                              onstant_tsc a

Partições do sistema de arquivos do sistema operacional :

In [4]:
!df -h

Sist. Arq.       Tam. Usado Disp. Uso% Montado em
udev             3,7G     0  3,7G   0% /dev
tmpfs            768M  1,9M  766M   1% /run
/dev/nvme0n1p6    18G   16G  915M  95% /
tmpfs            3,8G   83M  3,7G   3% /dev/shm
efivarfs         268K  148K  116K  57% /sys/firmware/efi/efivars
tmpfs            5,0M   12K  5,0M   1% /run/lock
tmpfs            1,0M     0  1,0M   0% /run/credentials/systemd-journald.service
/dev/nvme0n1p9   2,7G   20M  2,6G   1% /tmp
/dev/nvme0n1p7   6,4G  5,7G  390M  94% /var
/dev/nvme0n1p10  192G  8,3G  174G   5% /home
/dev/nvme0n1p5   975M  9,1M  966M   1% /boot/efi
tmpfs            768M  100K  767M   1% /run/user/1000


Memória RAM total e em uso pelo sistema operacional :

In [5]:
!free

               total       usada       livre    compart.  buff/cache  disponível
Mem.:        7854544     3540604     1898632      584460     3294724     4313940
Swap:        8104956           0     8104956


Versão do compilador C/C++ gcc :

In [6]:
!gcc --version

gcc (Debian 14.2.0-19) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



O módulo Pyton "platform" fornece diversas informações do sistema (computador, sistema operacional, Python, etc).

In [7]:
import platform

In [8]:
platform.platform()

'Linux-6.12.96+deb13-amd64-x86_64-with-glibc2.41'

Mais detalhes, como nome do computador na rede (node), arquitetura de hardware (32/64 bits), etc:

In [9]:
platform.uname()

uname_result(system='Linux', node='rfsDebian', release='6.12.96+deb13-amd64', version='#1 SMP PREEMPT_DYNAMIC Debian 6.12.96-1 (2026-07-20)', machine='x86_64')

### Informações sobre Python e módulos

#### Python

Número da versão de Python:

In [10]:
platform.python_version()

'3.13.5'

data da versão:

In [11]:
platform.python_build()

('main', 'Jul 15 2026 20:25:40')

compilador C/C++ utilizado para criar tal versão de Python :

In [12]:
platform.python_compiler()

'GCC 14.2.0'

In [13]:
import pandas as pd

In [14]:
pd.__version__

'3.0.3'

In [15]:
import numba as nb

In [16]:
nb.__version__

'0.66.0'

In [17]:
import numpy as np

In [18]:
np.__version__

'2.4.6'

---
## <font color="#5EAAE8">Inicialização do Problema</font>

In [19]:
!pip install astroquery

In [20]:
from numba import njit
from astroquery.jplhorizons import Horizons
from astropy.time import Time
import numpy as np
import pandas as pd

In [21]:
espacamento = 10
passos = 5000
g = 4 * np.pi**2 # em AU/ano
epsilon = 0.001
dt = 0.001
numCorpos = 9
frequenciaSnapshots = 20

A função calculaForcas calcula o valor das forças do sistema utilizando a fórmula:

$$ F_i = ∑_{\substack{j = 1 \\ j \neq i}}^{N} \frac{G m_{i} m_{j}}{|\vec{r}_{ij}|^{2}}$$

A função calculaEnergiaK calcula o valor da energia cinética utilizando a fórmula:

$$ K = ∑^{N}_{i = 1}\frac{m_{i}|\vec{v}_{i}|^{2}}{2}$$

A função calculaEnergiaU calcula o valor da energia cinética utilizando a fórmula:

$$ U = ∑^{N}_{i = 1}∑^{N}_{\substack{j = 1 \\ j \neq i}}\frac{G m_{i} m_{j}}{|r_{ij}|}$$

---
## <font color="#1B5A9D">Leitura dos dados</font>

In [22]:
tempoAlvo = Time('2026-07-24T20:00:00', format='isot', scale='utc')

#Ids dos corpos no Horizons
BODIES = {
    'Sol': 10,
    'Mercúrio': 199,
    'Vênus': 299,
    'Terra': 399,
    'Marte': 499,
    'Júpiter': 599,
    'Saturno': 699,
    'Urano': 799,
    'Netuno': 899
}

#Massas relativas ao Sol
MASSAS_SOLARES = {
    'Sol': 1.0,
    'Mercúrio': 1.660e-7,
    'Vênus': 2.447e-6,
    'Terra': 3.003e-6,
    'Marte': 3.227e-7,
    'Júpiter': 9.546e-4,
    'Saturno': 2.858e-4,
    'Urano': 4.366e-5,
    'Netuno': 5.151e-5
}

In [23]:
#Escrita do arquivo HDF5

posicoes = [] # em AU
velocidades = [] # em AU/ano
massas = []
nomes = []

for nome, objId in BODIES.items():
  obj = Horizons(id=objId, location='500@10', epochs = tempoAlvo.jd)
  vec = obj.vectors()

  #Para as posições em AU
  pos = np.array([vec['x'][0], vec['y'][0], vec['z'][0]])

  #Para a velocidade
  vel = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * 365.25

  posicoes.append(pos)
  velocidades.append(vel)
  massas.append(MASSAS_SOLARES[nome])
  nomes.append(nome)

posicoes = np.array(posicoes)
velocidades = np.array(velocidades)
massas = np.array(massas)

df = pd.DataFrame({
    'x': posicoes[:, 0],
    'y': posicoes[:, 1],
    'z': posicoes[:, 2],
    'vx': velocidades[:, 0],
    'vy': velocidades[:, 1],
    'vz': velocidades[:, 2],
    'massa': massas
})

df['nome'] = nomes

df.to_hdf("Entrada.h5", key="Entrada", mode="w", format='table')

In [24]:
dados = pd.read_hdf("Entrada.h5")

---
## <font color="#4680AF">Funções de Snapshot</font>

In [25]:
def salvarHDF5(kInicial, uInicial, kFinal, uFinal, biblioteca, dimensao):
  df = pd.DataFrame({"EnergiaCineticaInicial":[kInicial], "EnergiaPotencialInicial":[uInicial], "EnergiaCineticaFinal":[kFinal], "EnergiaPotencialFinal":[uFinal]})
  df.to_hdf("SaidaTeste.h5", key=biblioteca + "/" + dimensao + "/Resultados", mode="a", append=True)

In [26]:
def salvarSnapshot(df, biblioteca, dimensao):
  df.to_hdf("SnapshotTeste.h5", key=biblioteca + "/" + dimensao + "/Snapshots", mode="a", append=True, format='table' )

---
## <font color="#33AAFF">Numba CPU</font>

---
### <font color="#E86C4A">3D</font>

In [27]:
x = dados["x"].to_numpy().copy()
y = dados["y"].to_numpy().copy()
z = dados["z"].to_numpy().copy()
vx = dados["vx"].to_numpy().copy()
vy = dados["vy"].to_numpy().copy()
vz = dados["vz"].to_numpy().copy()
m = dados["massa"].to_numpy().copy()

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

In [28]:
x = np.array(x, dtype=np.float64)
y = np.array(y, dtype=np.float64)
z = np.array(z, dtype=np.float64)
m = np.array(m, dtype=np.float64)
vx = np.array(vx, dtype=np.float64)
vy = np.array(vy, dtype=np.float64)
vz = np.array(vz, dtype=np.float64)

forcaX = np.array(forcaX, dtype=np.float64)
forcaY = np.array(forcaY, dtype=np.float64)
forcaZ = np.array(forcaZ, dtype=np.float64)

In [29]:
@njit(cache=True)
def calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy
      forcaZ[j] += -forcaAtual * dz

In [30]:
@njit(cache=True)
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ):
    for k in range(numCorpos):
      forcaX[k] = 0.0
      forcaY[k] = 0.0
      forcaZ[k] = 0.0

    calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ)
    for j in range(numCorpos):

      acelX = forcaX[j] / m[j]
      acelY = forcaY[j] / m[j]
      acelZ = forcaZ[j] / m[j]

      vx[j] = vx[j] + acelX * dt
      vy[j] = vy[j] + acelY * dt
      vz[j] = vz[j] + acelZ * dt

      x[j] = x[j] + vx[j] * dt
      y[j] = y[j] + vy[j] * dt
      z[j] = z[j] + vz[j] * dt

In [31]:
@njit(cache=True)
def calculaEnergiaK(m, vx, vy, vz, numCorpos):
  k = 0.0
  for i in range(numCorpos):
    k += 0.5 * m[i] * (vx[i]*vx[i] + vy[i]*vy[i] + vz[i]*vz[i])
  return k

In [32]:
@njit(cache=True)
def calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon):
  u = 0.0
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)**0.5

      u += - (g * m[i] * m[j]) / dist

  return u

Tempo de Execução:

In [33]:
%time kInicial = calculaEnergiaK(m, vx, vy, vz, numCorpos)

CPU times: user 93.9 ms, sys: 3.84 ms, total: 97.7 ms
Wall time: 95.6 ms


In [34]:
%time uInicial = calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon)

CPU times: user 2.35 ms, sys: 0 ns, total: 2.35 ms
Wall time: 1.89 ms


In [35]:
print("Energia Mecânica Inicial: ", kInicial + uInicial)

Energia Mecânica Inicial:  -0.004428335207274474


In [36]:
snapshots = np.empty([len(range(0, passos, frequenciaSnapshots)) * numCorpos, 8], dtype=np.float64)
contSnap = 1

In [37]:
%%time
for i in range(passos):
  if(i % frequenciaSnapshots == 0):
      inicio = contSnap * numCorpos - numCorpos
      fim = contSnap * numCorpos
      contSnap += 1
      snapshots[inicio:fim, 0:8] = np.array([x, y, z, vx, vy,
                                             vz, m, np.full(numCorpos, i)]).transpose()

  movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 10.3 ms, sys: 0 ns, total: 10.3 ms
Wall time: 10 ms


In [38]:
df = pd.DataFrame(
    snapshots,
    columns=['x', 'y', 'z', 'vx', 'vy', 'vz', 'm', 'passo']
)

salvarSnapshot(df, biblioteca="NumbaCPU", dimensao="D3")

In [39]:
%time kFinal = calculaEnergiaK(m, vx, vy, vz, numCorpos)

CPU times: user 7 μs, sys: 0 ns, total: 7 μs
Wall time: 7.87 μs


In [40]:
%time uFinal = calculaEnergiaU(numCorpos, x, y, z, m, g, epsilon)

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 6.68 μs


In [41]:
salvarHDF5(kInicial, uInicial, kFinal, uFinal, biblioteca="NumbaCPU", dimensao="D3")

In [42]:
print("Energia Mecânica Final: ", kFinal + uFinal)

Energia Mecânica Final:  -0.004428441553657843


---
## <font color="#40BCD8">Resultados</font>

---
### <font color="#33AAFF">Numba CPU</font>

---
#### <font color="#E86C4A">3D</font>

In [43]:
df = pd.read_hdf("SaidaTeste.h5", "NumbaCPU/D3/Resultados")

In [44]:
df.EnergiaCineticaInicial

0    0.00433
Name: EnergiaCineticaInicial, dtype: float64

In [45]:
df.EnergiaPotencialInicial

0   -0.008758
Name: EnergiaPotencialInicial, dtype: float64

In [46]:
df.EnergiaCineticaFinal

0    0.004409
Name: EnergiaCineticaFinal, dtype: float64

In [47]:
df.EnergiaPotencialFinal

0   -0.008838
Name: EnergiaPotencialFinal, dtype: float64